In [2]:
pip install yfinance

In [55]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm


def obtener_variacion_logaritmica_porcentaje(ticker):
    """
    Esta función devuelve un DataFrame con las variaciones logarítmicas de los precios de cierre
    de un ticker dado para un rango de fechas, expresadas en porcentaje y con el formato decimal ajustado
    para usar comas como separadores decimales, además de añadir el símbolo de porcentaje.

    Parámetros:
    - ticker: El símbolo del ticker de la acción (como string).
    - fecha_inicio: La fecha de inicio del rango en formato 'AAAA-MM-DD'.
    - fecha_fin: La fecha de fin del rango en formato 'AAAA-MM-DD'.
    """
    fecha_inicio = "2018-01-01"
    fecha_fin = "2023-12-31"
    # Descargar los datos del ticker
    datos = yf.download(ticker, start=fecha_inicio, end=fecha_fin)

    # Seleccionar solo la columna 'Close'
    precios_cierre = datos[['Close']]

    # Calcular la variación logarítmica de los precios de cierre
    variacion_log = np.log(precios_cierre / precios_cierre.shift(1))

    # Convertir la variación logarítmica a formato porcentual
    variacion_log_porcentaje = variacion_log * 100

    # Convertir a string, usar coma como separador decimal y añadir el símbolo de porcentaje
    variacion_log_porcentaje = variacion_log_porcentaje['Close'].replace('.', ',')

    # Crear un nuevo DataFrame para devolver, usando la fecha como índice
    df_resultado = pd.DataFrame(variacion_log_porcentaje)
    df_resultado.rename(columns={'Close': 'Variacion Logaritmica (%)'}, inplace=True)

    return df_resultado



[*********************100%%**********************]  1 of 1 completed


,Variacion Logaritmica (%)
Date,
2018-01-02,NaN
2018-01-03,-1.028583
2018-01-04,-0.832453
2018-01-05,0.621042
2018-01-08,6.075470


In [57]:
# Ejemplo de uso
ticker = "TSLA"  # Símbolo del ticker para Apple Inc.


df_variacion_log = obtener_variacion_logaritmica_porcentaje(ticker)
df_variacion_log.head()


famafrench_df = pd.read_csv('/content/csv_general.csv', sep = ';')


# Convierte la columna de fecha a datetime
famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])

# Si necesitas un formato específico, puedes usar el parámetro 'format'
# df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y-%m-%d')

# Después de convertir, establece la columna de fecha como índice si deseas
famafrench_df.set_index('Date', inplace=True)



famafrench_df.head()

# Verifica los nombres de las columnas de ambos DataFrames
print(df_variacion_log.columns)
print(famafrench_df.columns)

# Si 'Date' ya es el índice o si la columna tiene otro nombre, ajusta el código.
# Supongamos que la fecha ya es el índice o tiene otro nombre, entonces puedes saltarte el paso de set_index

# Si ambos DataFrames tienen el índice de fecha correctamente configurado, puedes proceder directamente a merge
df_combinado = famafrench_df.merge(df_variacion_log, left_index=True, right_index=True, how='outer')
# Eliminar filas que contengan algún valor NaN
df_combinado = df_combinado.dropna()

# Convertir el índice de fecha a una columna regular

df_combinado = df_combinado.round(3)



print(df_combinado)
print(df_combinado.dtypes)

# Asegúrate de que las columnas son tratadas como strings antes de reemplazar ',' por '.'
df_combinado['Mkt-RF'] = df_combinado['Mkt-RF'].astype(str).str.replace(',', '.').astype(float)
df_combinado['SMB'] = df_combinado['SMB'].astype(str).str.replace(',', '.').astype(float)
df_combinado['HML'] = df_combinado['HML'].astype(str).str.replace(',', '.').astype(float)


# Asegura que Pandas trate las columnas como strings antes de realizar operaciones de strings
df_combinado['RF'] = df_combinado['RF'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100
df_combinado['Variacion Logaritmica (%)'] = df_combinado['Variacion Logaritmica (%)'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100

# Para % Fundflows, asume que ya has manejado los NaNs o que están siendo manejados de manera implícita aquí
df_combinado['% Fundflows'] = df_combinado['% Fundflows'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100

# Verifica los cambios
print(df_combinado.head())
print(df_combinado.dtypes)


# Preparar las variables independientes
# Añadir una constante al modelo para el término de intercepción
X = df_combinado[['Mkt-RF', 'SMB', 'HML', '% Fundflows']]
X = sm.add_constant(X)

# La variable dependiente es el exceso de rendimiento de Apple
# Asegúrate de que 'RF' y 'rendimientos apple' están en formato decimal adecuado
Y = df_combinado['Variacion Logaritmica (%)'] - df_combinado['RF']

# Estimar el modelo OLS
modelo = sm.OLS(Y, X).fit()

# Mostrar el resumen del modelo
print(modelo.summary())




[*********************100%%**********************]  1 of 1 completed

Index(['Variacion Logaritmica (%)'], dtype='object')
Index(['Mkt-RF', 'SMB', 'HML', 'RF', '% Fundflows'], dtype='object')
           Mkt-RF    SMB    HML     RF % Fundflows  Variacion Logaritmica (%)
Date                                                                         
2018-01-03  -1,19   1,01  -0,02  0,50%       0,13%                     -1.029
2018-01-05   0,24   0,22  -0,54  0,60%      -0,10%                      0.621
2018-01-08  -0,13   0,04  -0,19  0,70%       0,65%                      6.075
2018-01-10   0,16  -1,62   0,34  0,80%      -0,28%                      0.332
2018-01-11   1,29   1,35  -1,14  0,80%      -0,52%                      0.936
...           ...    ...    ...    ...         ...                        ...
2023-12-21   0,34  -0,43  -0,88  2,10%       0,25%                      2.935
2023-12-22   1,55   1,72   1,35  2,10%       0,28%                     -0.773
2023-12-26   0,51   1,12   2,05  2,10%       0,49%                      1.599
2023-12-27  -0,06   


<ipython-input-57-41357d6b1d12>:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])


In [34]:
df_combinado.to_csv('ex.csv', sep = ";", index=True)